# Random Forest 
**IFRI AI Classes**

---

The Random Forest algorithm is an ensemble learning method that constructs multiple decision trees during training and merges their outputs for prediction. Introduced by Leo Breiman in 2001, it combines bagging and random subspace selection to create a powerful and robust model for classification and regression tasks.


## 1. Key Concepts

Random Forest is an ensemble of decision trees, where multiple trees are trained independently and their predictions are aggregated. The two key sources of randomness that make Random Forest effective are bagging and random feature subset selection.

### Bagging (Bootstrap Aggregating)

Each tree is trained on a bootstrap sample $D_i$ drawn with replacement from the original training set $D$:

$$D = \{(x_1, y_1), (x_2, y_2), \ldots, (x_N, y_N)\}$$


$$D_i \sim \text{Bootstrap}(D) \quad \text{where} \quad |D_i| = N$$

Since sampling is done with replacement, each bootstrap sample contains approximately 63.2% unique training instances, while the remaining 36.8% are duplicates.

### Random Feature Subset Selection

When building each tree and choosing the best split at any node, only a random subset of m features out of $p$ total features is considered:

**For classification**:

$$m = \lfloor \sqrt{p} \rfloor$$

**For regression**:

$$m = \left\lfloor \frac{p}{3} \right\rfloor$$

This further decorrelates the trees by preventing strong predictors from dominating every split.

### Random Forest Example

![Random Forest illustration](https://miro.medium.com/v2/resize:fit:1400/1*i0o8mjFfCn-uD79-F1Cqkw.png)

*Figure 1: Illustration of the Random Forest algorithm. Multiple decision trees are trained on bootstrap samples with random feature subsets. The final prediction is the majority vote for classification or the average for regression.*

Source: *Understanding Random Forest, Towards Data Science*

### Ensemble Prediction

The final prediction is obtained by aggregating all tree predictions.

**For classification**:

$$\hat{y} = \text{mode}\{h_1(x), h_2(x), \ldots, h_B(x)\}$$

**For regression**:

$$\hat{y} = \frac{1}{B} \sum_{b=1}^{B} h_b(x)$$

Where $B$ is the number of trees and $h_b(x)$ is the prediction of tree $b$ for input $x$.

### Out-of-Bag Error Estimation

The samples not included in a tree's bootstrap sample can be used as an internal validation set. The out-of-bag error provides an unbiased estimate of the generalization error without a separate validation set:

$$\text{OOB Error} = \frac{1}{N} \sum_{i=1}^{N} L(y_i, \hat{y}_{\text{OOB}, i})$$

## 2. Impurity Measures

Decision trees split nodes by maximizing impurity reduction. The choice of impurity measure depends on the task.

### Gini Impurity (Classification)

For a node with K classes:

$$\operatorname{Gini}(t) = 1 - \sum_{k=1}^{K} p_k^2$$

Where $p_k$ is the proportion of class $k$ samples in node $t$.

**Gini gain** for splitting node $t$ into left and right children is:

$$\operatorname{Gain} = \operatorname{Gini}(t) - \left(\frac{n_L}{n} \operatorname{Gini}(t_L) + \frac{n_R}{n} \operatorname{Gini}(t_R)\right)$$

### Mean Squared Error (Regression)

For regression tasks, variance reduction is used:

$$\operatorname{MSE}(t) = \frac{1}{n} \sum_{i \in t} (y_i - \bar{y}_t)^2$$

The gain is computed similarly:

$$\operatorname{Gain} = \operatorname{MSE}(t) - \left(\frac{n_L}{n} \operatorname{MSE}(t_L) + \frac{n_R}{n} \operatorname{MSE}(t_R)\right)$$

### How Splits Are Chosen

For each candidate split defined by a feature index and threshold value:
- Partition the node's samples into left and right subsets.
- Calculate the weighted average impurity of the children.
- Select the split that maximizes impurity reduction.

This greedy procedure continues recursively until a stopping criterion is met.


## 3. Pseudo-algorithm


D ← training set of n labeled instances (x_i, y_i)

B ← number of trees in the forest

m ← number of features to consider per split

function RandomForest_Fit(D, B, m)
    trees ← []
    feature_indices ← []

    for b = 1 to B do
        D_boot ← Bootstrap sample from D (n samples with replacement)
        feat_idx ← Random sample of m features from {1, 2, ..., p} without replacement

        tree_b ← build_tree(D_boot, feat_idx, depth=0, max_depth=5)

        append tree_b to trees
        append feat_idx to feature_indices
    end for

    return (trees, feature_indices)
end

function build_tree(D_subset, feat_idx, depth, max_depth)
    if (depth = max_depth) or (|D_subset| < min_samples_split) or (all y_i identical)
        return leaf_value(D_subset)
    end if

    best_gain ← -∞
    best_feat, best_thresh ← None, None

    for each feature j in feat_idx do
        for each unique threshold t in D_subset[:, j] do
            D_left  ← samples where x[j] ≤ t
            D_right ← samples where x[j] > t

            if D_left is empty or D_right is empty: continue

            gain ← impurity_gain(D_subset, D_left, D_right)

            if gain > best_gain then
                best_gain ← gain
                best_feat ← j
                best_thresh ← t
            end if
        end for
    end for

    if best_feat is None then
        return leaf_value(D_subset)
    end if

    left_tree  ← build_tree(D_left, feat_idx, depth + 1, max_depth)
    right_tree ← build_tree(D_right, feat_idx, depth + 1, max_depth)

    return (best_feat, best_thresh, left_tree, right_tree)
end

function RandomForest_Predict(x, forest)
    (trees, feature_indices) ← forest

    predictions ← []
    for each (tree, feat_idx) in zip(trees, feature_indices) do
        pred ← traverse_tree(x[feat_idx], tree)
        append pred to predictions
    end for

    if task is classification:
        return mode(predictions)
    else:
        return mean(predictions)
end


### Training in the *ifri_mini_ml_lib* implementation:

In the `fit` method, each tree is built independently using a bootstrap sample. The `_bootstrap_sample` method randomly draws $n_{\text{samples}}$ instances with replacement, creating the diversity necessary for the ensemble to work. A random subset of features is selected for each tree via `_get_max_features`, which supports `'sqrt'`, `'log2'`, integer, or float specifications. Each internal `_DecisionTree` is then trained on its specific data and feature subset, with task set to `'classify'` for classification and `'regress'` for regression. The `predict` method calls `predict` on each tree using only the feature subset that tree was trained with, then aggregates predictions by mode for classification or mean for regression.


## 4. Implementation

For this implementation, we will use the Iris dataset for classification and a generated regression dataset for regression.


In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from ifri_mini_ml_lib.preprocessing.preparation import DataSplitter

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target)

X.head()


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [2]:
# Split the data into training and testing sets
splitter = DataSplitter(seed=42)
X_train, X_test, y_train, y_test = splitter.train_test_split(X, y, test_size=0.2)


**Regression with Scikit-learn**


In [3]:
import time
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor

start_r1 = time.perf_counter()
rf_sklearn = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_sklearn.fit(X_train, y_train)
pred_sklearn = rf_sklearn.predict(X_test)
end_r1 = time.perf_counter()

**Regression with ifri_mini_ml_lib**


In [8]:
from ifri_mini_ml_lib.regression import RandomForestRegressor

start_2 = time.perf_counter()
rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=4,
    max_features=2
)
rf.fit(X_train.values, y_train.values)
end_2 = time.perf_counter()

In [15]:
# Predict the target values for the test set
y_pred_1 = rf_sklearn.predict(X_test)
y_pred_2 = rf.predict(X_test.values)

# Evaluate the model
from ifri_mini_ml_lib.metrics.regression import evaluate_rg_model

metrics_1 = evaluate_rg_model(list(y_test), list(y_pred_1))
metrics_2 = evaluate_rg_model(list(y_test), list(y_pred_2))

reg_results = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'MAE', 'MAPE', 'R²', 'Time (s)'],
    'Scikit-learn': [
        f"{metrics_1['MSE']:.4f}",
        f"{metrics_1['RMSE']:.4f}",
        f"{metrics_1['MAE']:.4f}",
        f"{metrics_1['MAPE']:.4f}",
        f"{metrics_1['R²']:.4f}",
        f"{end_r1 - start_r1:.4f}"
    ],
    'ifri_mini_ml_lib': [
        f"{metrics_2['MSE']:.4f}",
        f"{metrics_2['RMSE']:.4f}",
        f"{metrics_2['MAE']:.4f}",
        f"{metrics_2['MAPE']:.4f}",
        f"{metrics_2['R²']:.4f}",
        f"{end_2 - start_2:.4f}"
    ],
})

reg_results

,Metric,Scikit-learn,ifri_mini_ml_lib
0,MSE,0.0014,0.0060
1,RMSE,0.0372,0.0776
2,MAE,0.0137,0.0426
3,MAPE,1.1333,3.0874
4,R²,0.9980,0.9914
5,Time (s),0.4109,1.1372


The *ifri_mini_ml_lib* implementation achieves competitive performance while being faster due to its minimal overhead and direct NumPy array operations. The slightly lower accuracy is expected because the Scikit-learn version uses optimized splitting algorithms and additional features such as feature importance and more sophisticated tree building.


## 5. Interactive Demos


### Regression Demo


In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown
import matplotlib.pyplot as plt
import numpy as np

def plot_random_forest_regression(n_estimators=50, max_depth=5, noise=0.2, library='ifri_mini_ml_lib'):
    """
    Interactive Random Forest regression plot with noise parameter.
    """
    # Generate noisy regression data
    from sklearn.datasets import make_regression
    X_reg, y_reg = make_regression(n_samples=200, n_features=1, noise=noise * 50, random_state=42)
    
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
    
    if library == 'sklearn':
        from sklearn.ensemble import RandomForestRegressor
        model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    else:
        from ifri_mini_ml_lib.regression import RandomForestRegressor
        model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, max_features=1)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    from ifri_mini_ml_lib.metrics.regression import evaluate_rg_model
    metrics = evaluate_rg_model(y_test.tolist(), preds.tolist())

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(X_test, y_test, alpha=0.6, edgecolors='k', s=40, label='True values')
    ax.scatter(X_test, preds, alpha=0.6, s=40, marker='x', label='Predictions')
    ax.set_xlabel('Feature')
    ax.set_ylabel('Target')
    ax.set_title(f'Random Forest ({library}) — R²: {metrics["R²"]:.3f} | RMSE: {metrics["RMSE"]:.3f}')
    ax.legend()
    plt.tight_layout()
    plt.show()

interact(
    plot_random_forest_regression,
    n_estimators=IntSlider(min=1, max=200, step=10, value=50, description='Trees:'),
    max_depth=IntSlider(min=1, max=15, step=1, value=5, description='Max Depth:'),
    noise=FloatSlider(min=0.0, max=1.0, step=0.1, value=0.2, description='Noise:'),
    library=Dropdown(
        options=['sklearn', 'ifri_mini_ml_lib'],
        value='ifri_mini_ml_lib',
        description='Library:'
    )
);

interactive(children=(IntSlider(value=50, description='Trees:', max=200, min=1, step=10), IntSlider(value=5, d…

The regression demo shows how the Random Forest fits noisy data. Each individual tree's prediction is displayed as a thin transparent line, while the ensemble average is shown as a thick red line. This illustrates how averaging multiple trees reduces variance and produces a smoother fit than any single tree.


## 6. Real-life Applications

Random Forest is one of the most widely used algorithms in industry due to its robustness, interpretability, and strong out-of-the-box performance.

- **Credit risk assessment and fraud detection**: Banks and financial institutions use Random Forest to predict loan default probability and detect fraudulent transactions.
- **Medical diagnosis and disease prediction**: Researchers apply Random Forest to diagnose diseases from clinical data, genetic markers, or medical images.
- **Customer churn prediction**: Telecom companies and subscription services predict which customers are likely to leave.
- **Remote sensing and land cover classification**: Satellite imagery classification benefits from Random Forest's ability to handle many spectral bands and resist overfitting.
- **Drug discovery and computational chemistry**: Random Forest predicts molecular properties, biological activity, and toxicity from chemical structures.
- **Predictive maintenance in manufacturing**: Industrial equipment sensors generate time-series data used to predict failures before they occur.

### Random Forest excels when:
- High prediction accuracy is needed with minimal tuning.
- Mixed data types are present.
- Feature importance interpretation is valuable.
- The relationship between features and target is non-linear.
- The dataset contains noise or outliers.
- Parallelism can be leveraged for training speed.


## 7. Limitations and Challenges

While Random Forest is remarkably effective, it has important limitations that practitioners must understand.

- **Lack of interpretability**: an ensemble of many trees becomes much harder to explain than a single decision tree.
- **Computational cost at training time**: building many trees can be expensive on large datasets.
- **Memory requirements**: storing hundreds of trees can consume substantial memory.
- **Poor extrapolation in regression**: predictions remain within the range of the training targets.
- **Bias toward high-cardinality categorical features**: one-hot encoded features with many categories may receive undue preference.
- **Sensitivity to hyperparameters on small datasets**: small datasets can produce unstable results with poor settings.
- **Not competitive on text or image data**: deep learning often performs better on raw unstructured data.
- **Correlated trees may limit variance reduction**: strong global predictors can make trees more similar.

Overall, Random Forest remains one of the best off-the-shelf algorithms for structured or tabular data.


## 8. References

- Random Forests, Leo Breiman, *Machine Learning*, 45(1):5-32, 2001, [https://link.springer.com/article/10.1023/A:1010933404324](https://link.springer.com/article/10.1023/A:1010933404324)
- The Elements of Statistical Learning, Trevor Hastie, Robert Tibshirani, Jerome Friedman, Chapter 15, Springer, 2009, [https://hastie.su.domains/ElemStatLearn/](https://hastie.su.domains/ElemStatLearn/)
- Random Forest Classifier, Scikit-learn Documentation, [https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- Random Forest Regressor, Scikit-learn Documentation, [https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)
- Ensemble Methods, Scikit-learn User Guide, [https://scikit-learn.org/stable/modules/ensemble.html#forest](https://scikit-learn.org/stable/modules/ensemble.html#forest)
- Understanding Random Forests, Gilles Louppe, PhD Thesis, 2014, [https://arxiv.org/abs/1407.7502](https://arxiv.org/abs/1407.7502)
- Classification and Regression by RandomForest, Andy Liaw and Matthew Wiener, *R News*, 2002, [https://www.r-project.org/doc/Rnews/Rnews_2002-3.pdf](https://www.r-project.org/doc/Rnews/Rnews_2002-3.pdf)
- Do We Need Hundreds of Classifiers to Solve Real World Classification Problems?, Manuel Fernández-Delgado et al., *JMLR*, 2014, [https://jmlr.org/papers/v15/delgado14a.html](https://jmlr.org/papers/v15/delgado14a.html)
